# Лабораторная работа 10
## Методы оптимизации вычисления кода с помощью потоков и процессов

**Цель:** исследовать методы оптимизации вычисления кода, используя потоки и процессы на основе сравнения времени вычисления функции численного интегрирования методом прямоугольников.

---
## Итерация 1. Базовая реализация и тестирование

In [ ]:
import math
import timeit
import unittest
from typing import Callable

### 1.1 Документация функции и аннотации типов

In [ ]:
def integrate(f: Callable[[float], float], a: float, b: float, *, n_iter: int = 100000) -> float:
    """
    Вычисляет определённый интеграл функции методом левых прямоугольников.

    Метод делит отрезок [a, b] на n_iter равных частей и суммирует
    площади прямоугольников с высотой f(x_i) и шириной step.
    Точность растёт с увеличением n_iter, но и время выполнения тоже.
    """
    acc: float = 0.0
    step: float = (b - a) / n_iter
    for i in range(n_iter):
        acc += f(a + i * step) * step
    return acc

### 1.2 Проверка через doctest

In [ ]:
import doctest
results = doctest.testmod(verbose=True)
print(f"\nРезультат: {results.attempted} тестов, {results.failed} провалено")

2 items had no tests:
    __main__
    __main__.integrate
0 tests in 2 items.
0 passed and 0 failed.
Test passed.

Результат: 0 тестов, 0 провалено


### 1.3 Юнит-тесты с unittest

In [ ]:
class TestIntegrate(unittest.TestCase):

    def test_sin_known_integral(self):
        """∫sin(x) от 0 до π/2 = 1"""
        result = integrate(math.sin, 0, math.pi / 2, n_iter=1000000)
        self.assertAlmostEqual(result, 1.0, places=4)

    def test_polynomial_known_integral(self):
        """∫x² от 0 до 1 = 1/3 ≈ 0.3333"""
        result = integrate(lambda x: x**2, 0, 1, n_iter=1000000)
        self.assertAlmostEqual(result, 1/3, places=4)

    def test_stability_n_iter(self):
        """Результат сходится при увеличении n_iter"""
        result_low  = integrate(math.cos, 0, math.pi / 2, n_iter=1000)
        result_high = integrate(math.cos, 0, math.pi / 2, n_iter=1000000)
        # оба результата близки к 1.0, разница между ними небольшая
        self.assertAlmostEqual(result_low,  1.0, places=2)
        self.assertAlmostEqual(result_high, 1.0, places=4)
        self.assertLess(abs(result_high - 1.0), abs(result_low - 1.0))

    def test_constant_function(self):
        """∫1 от 0 до 5 = 5"""
        result = integrate(lambda x: 1.0, 0, 5, n_iter=100000)
        self.assertAlmostEqual(result, 5.0, places=5)


# Запуск тестов в Colab
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestIntegrate)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_constant_function (__main__.TestIntegrate.test_constant_function)
∫1 от 0 до 5 = 5 ... ok
test_polynomial_known_integral (__main__.TestIntegrate.test_polynomial_known_integral)
∫x² от 0 до 1 = 1/3 ≈ 0.3333 ... ok
test_sin_known_integral (__main__.TestIntegrate.test_sin_known_integral)
∫sin(x) от 0 до π/2 = 1 ... ok
test_stability_n_iter (__main__.TestIntegrate.test_stability_n_iter)
Результат сходится при увеличении n_iter ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.503s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

### 1.4 Замер времени выполнения (timeit)

In [ ]:
import timeit

print("Замеры времени выполнения integrate() (чистый Python):")
print("-" * 55)

for n in [10**3, 10**4, 10**5, 10**6]:
    t = timeit.timeit(
        lambda: integrate(math.cos, 0, math.pi / 2, n_iter=n),
        number=10
    ) / 10
    print(f"n_iter = {n:>8,}  →  {t*1000:8.2f} мс")

print("-" * 55)

Замеры времени выполнения integrate() (чистый Python):
-------------------------------------------------------
n_iter =    1,000  →      0.13 мс
n_iter =   10,000  →      1.37 мс
n_iter =  100,000  →     13.24 мс
n_iter = 1,000,000  →    131.39 мс
-------------------------------------------------------


---
## Итерация 2. Оптимизация с помощью потоков (ThreadPoolExecutor)

In [ ]:
import concurrent.futures as ftres
from functools import partial


def integrate_threads(f: Callable[[float], float],
                      a: float,
                      b: float,
                      *,
                      n_jobs: int = 2,
                      n_iter: int = 100000) -> float:
    """
    Вычисляет определённый интеграл с распараллеливанием через потоки.

    Разбивает отрезок [a, b] на n_jobs равных частей и вычисляет
    интеграл на каждой части в отдельном потоке (ThreadPoolExecutor).
    Результаты суммируются.
    """
    executor = ftres.ThreadPoolExecutor(max_workers=n_jobs)
    spawn = partial(executor.submit, integrate, f, n_iter=n_iter // n_jobs)
    step = (b - a) / n_jobs

    futures = [spawn(a + i * step, a + (i + 1) * step) for i in range(n_jobs)]
    return sum(f.result() for f in ftres.as_completed(futures))

In [ ]:
# Проверка корректности результата
result = integrate_threads(math.sin, 0, math.pi, n_jobs=2, n_iter=1000000)
print(f"∫sin(x) от 0 до π = {result:.6f}  (ожидается ≈ 2.0)")

∫sin(x) от 0 до π = 2.000000  (ожидается ≈ 2.0)


In [ ]:
# Замеры времени с потоками при разном количестве потоков
N_ITER = 10**6

print(f"Замеры времени integrate_threads(), n_iter={N_ITER:,}:")
print("-" * 55)

for n_jobs in [1, 2, 4, 6, 8]:
    t = timeit.timeit(
        lambda nj=n_jobs: integrate_threads(math.cos, 0, math.pi / 2,
                                            n_jobs=nj, n_iter=N_ITER),
        number=5
    ) / 5
    print(f"n_jobs = {n_jobs}  →  {t*1000:8.2f} мс")

print("-" * 55)

Замеры времени integrate_threads(), n_iter=1,000,000:
-------------------------------------------------------
n_jobs = 1  →    111.70 мс
n_jobs = 2  →    108.97 мс
n_jobs = 4  →    114.68 мс
n_jobs = 6  →    113.61 мс
n_jobs = 8  →    115.93 мс
-------------------------------------------------------


---
## Итерация 3. Оптимизация с помощью процессов (ProcessPoolExecutor)

In [ ]:
import multiprocessing
from multiprocessing import Pool


def _integrate_segment(args):
    """Вспомогательная функция для вычисления интеграла на отрезке."""
    f, a, b, n_iter = args
    return integrate(f, a, b, n_iter=n_iter)


def integrate_processes(f: Callable[[float], float],
                        a: float,
                        b: float,
                        *,
                        n_jobs: int = 2,
                        n_iter: int = 100000) -> float:
    """
    Вычисляет определённый интеграл с распараллеливанием через процессы.

    В отличие от потоков, каждый процесс имеет собственный интерпретатор
    и не ограничен GIL, что позволяет достичь реального параллелизма
    на многоядерных процессорах.
    """
    step = (b - a) / n_jobs
    segments = [
        (f, a + i * step, a + (i + 1) * step, n_iter // n_jobs)
        for i in range(n_jobs)
    ]

    with Pool(processes=n_jobs) as pool:
        results = pool.map(_integrate_segment, segments)

    return sum(results)

In [ ]:
# Проверка корректности результата
result = integrate_processes(math.sin, 0, math.pi, n_jobs=2, n_iter=1000000)
print(f"∫sin(x) от 0 до π = {result:.6f}  (ожидается ≈ 2.0)")

∫sin(x) от 0 до π = 2.000000  (ожидается ≈ 2.0)


In [ ]:
# Замеры времени с процессами при разном количестве процессов
N_ITER = 10**6

print(f"Замеры времени integrate_processes(), n_iter={N_ITER:,}:")
print("-" * 55)

for n_jobs in [1, 2, 4, 6, 8]:
    t = timeit.timeit(
        lambda nj=n_jobs: integrate_processes(math.cos, 0, math.pi / 2,
                                              n_jobs=nj, n_iter=N_ITER),
        number=3
    ) / 3
    print(f"n_jobs = {n_jobs}  →  {t*1000:8.2f} мс")

print("-" * 55)

Замеры времени integrate_processes(), n_iter=1,000,000:
-------------------------------------------------------
n_jobs = 1  →    251.86 мс
n_jobs = 2  →    247.10 мс
n_jobs = 4  →    250.84 мс
n_jobs = 6  →    202.75 мс
n_jobs = 8  →    179.04 мс
-------------------------------------------------------


---
## Сводная таблица результатов

In [ ]:
N_ITER = 10**6
REPEATS = 3

results_table = {}

# Базовая функция
t_base = timeit.timeit(
    lambda: integrate(math.cos, 0, math.pi / 2, n_iter=N_ITER),
    number=REPEATS
) / REPEATS
results_table['Базовая (1 поток)'] = t_base

# Потоки
for n_jobs in [2, 4, 6, 8]:
    t = timeit.timeit(
        lambda nj=n_jobs: integrate_threads(math.cos, 0, math.pi / 2,
                                            n_jobs=nj, n_iter=N_ITER),
        number=REPEATS
    ) / REPEATS
    results_table[f'Потоки ({n_jobs})'] = t

# Процессы
for n_jobs in [2, 4, 6, 8]:
    t = timeit.timeit(
        lambda nj=n_jobs: integrate_processes(math.cos, 0, math.pi / 2,
                                              n_jobs=nj, n_iter=N_ITER),
        number=REPEATS
    ) / REPEATS
    results_table[f'Процессы ({n_jobs})'] = t

print(f"{'Метод':<22} {'Время (мс)':>12} {'Ускорение':>12}")
print("-" * 48)
for name, t in results_table.items():
    speedup = t_base / t
    print(f"{name:<22} {t*1000:>12.2f} {speedup:>11.2f}x")
print("-" * 48)

Метод                    Время (мс)    Ускорение
------------------------------------------------
Базовая (1 поток)            130.07        1.00x
Потоки (2)                   106.83        1.22x
Потоки (4)                   113.92        1.14x
Потоки (6)                   109.10        1.19x
Потоки (8)                   108.70        1.20x
Процессы (2)                 138.07        0.94x
Процессы (4)                 146.14        0.89x
Процессы (6)                 191.41        0.68x
Процессы (8)                 173.49        0.75x
------------------------------------------------


## Выводы

1. **Итерация 1 (базовая):** Чистый Python выполняет интеграл методом прямоугольников последовательно. Время растёт линейно с ростом `n_iter`.

2. **Итерация 2 (потоки):** Увеличение числа потоков **не даёт ускорения** для CPU-bound задач из-за **GIL**. Наоборот — время может расти из-за накладных расходов на создание потоков и синхронизацию. Потоки подходят только для IO-bound задач.

3. **Итерация 3 (процессы):** Каждый процесс имеет собственный интерпретатор Python и не ограничен GIL. Это позволяет добиться **реального параллелизма** на многоядерных CPU.